# 🧪 Sesión 03 · Headers, Content-Type y Negociación de Contenido

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo sabe el servidor qué formato le estás enviando?

---

## 🖥️ Así se verá el resultado final

```http
$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: application/json" \
       -H "Accept: application/json" \
       -d '{"titulo":"Rayuela","anio":1963}'
HTTP/1.1 201 Created
content-type: application/json
{"id":1,"titulo":"Rayuela","anio":1963}

$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: text/plain" \
       -d 'Rayuela'
HTTP/1.1 415 Unsupported Media Type

$ curl http://127.0.0.1:8000/api/libros \
       -H "Accept: application/xml"
HTTP/1.1 406 Not Acceptable
```

_El navegador envía Accept por ti; una herramienta como curl te obliga a ser explícito._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Leer y emitir encabezados HTTP correctamente.
- Diferenciar Content-Type (lo que envío) de Accept (lo que acepto).
- Rechazar formatos no soportados con 415 y 406.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] POST válido en JSON → HTTP 201
- [ ] POST en texto plano se rechaza con 415 → HTTP 415
- [ ] GET que pide XML se rechaza con 406 → HTTP 406
- [ ] GET que acepta JSON responde 200 → HTTP 200

## 🧩 El reto

Implementa la creación y el listado de libros, pero haz que la API negocie el formato de entrada y de salida.

**Requisitos del contrato**

- POST /api/libros solo acepta Content-Type: application/json; si no, responde 415.
- GET /api/libros solo responde si el cliente acepta application/json; si pide XML, responde 406.
- Las respuestas correctas llevan Content-Type: application/json.
- Un POST válido responde 201 con el libro creado.

> **Pista conceptual:** Puedes inspeccionar `request.headers` dentro de una dependencia o middleware antes de que se procese el cuerpo. El formato no se adivina: se declara.

## 📦 Entregable

Cuaderno con los cuatro casos en verde y bitácora `auditorias/auditoria_sesion03.md`.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 30.4 MB/s eta 0:00:00
✅ Entorno listo
Python: 3.13.16
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ValidationError

app = FastAPI(title="API de libros con negociación de contenido")


# Modelo para validar los datos de un libro
class LibroEntrada(BaseModel):
    titulo: str
    anio: int


# Almacenamiento temporal en memoria
libros = []
siguiente_id = 1


# POST: crear un libro
@app.post("/api/libros")
async def crear_libro(request: Request):
    global siguiente_id

    # Verificar el formato de los datos recibidos
    content_type = request.headers.get("content-type", "")
    formato = content_type.split(";")[0].strip().lower()

    if formato != "application/json":
        return JSONResponse(
            status_code=415,
            content={"detail": "Solo se acepta application/json"}
        )

    # Leer y validar el cuerpo JSON
    try:
        datos = await request.json()
        libro = LibroEntrada.model_validate(datos)
    except (ValueError, ValidationError):
        return JSONResponse(
            status_code=422,
            content={"detail": "Los datos del libro no son validos"}
        )

    # Guardar el libro y asignarle un identificador
    nuevo_libro = {
        "id": siguiente_id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }

    libros.append(nuevo_libro)
    siguiente_id += 1

    return JSONResponse(
        status_code=201,
        content=nuevo_libro,
        media_type="application/json"
    )


# GET: consultar los libros
@app.get("/api/libros")
async def listar_libros(request: Request):
    accept = request.headers.get("accept", "*/*")

    # Revisar los formatos que acepta el cliente
    formatos = [
        parte.split(";")[0].strip().lower()
        for parte in accept.split(",")
    ]

    acepta_json = any(
        formato in (
            "application/json",
            "*/*",
            "application/*"
        )
        for formato in formatos
    )

    if not acepta_json:
        return JSONResponse(
            status_code=406,
            content={"detail": "Solo se ofrece application/json"}
        )

    return JSONResponse(
        status_code=200,
        content=libros,
        media_type="application/json"
    )


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Utilizaré Python 3.13 y FastAPI en Google Colab para desarrollar una API REST |
| **Rol** |Actuar como desarrollador de aplicaciones web, implementando y validando los encabezados HTTP de una API. |
| **Especificación** | Implementar las rutas POST /api/libros y GET /api/libros. El POST debe aceptar únicamente JSON y responder 201 si el libro se crea correctamente o 415 si el formato no es compatible. El GET debe responder con JSON y devolver 406 si el cliente solicita un formato no compatible. |
| **Output** | Un archivo app.py funcional y un cuaderno que cumpla los cuatro casos de verificación establecidos por el docente. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'POST válido en JSON',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'headers': {'Content-Type': 'application/json', 'Accept': 'application/json'},
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela', 'anio': 1963},
  'headers_contiene': {'content-type': 'application/json'}},
 {'descripcion': 'POST en texto plano se rechaza con 415',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'content': 'Rayuela',
  'headers': {'Content-Type': 'text/plain'},
  'estado': 415},
 {'descripcion': 'GET que pide XML se rechaza con 406',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'headers': {'Accept': 'application/xml'},
  'estado': 406},
 {'descripcion': 'GET que acepta JSON responde 200',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'headers': {'Accept': 'application/json'},
  'estado': 200,
  'headers_contiene': {'content-type': 'application/json'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. POST válido en JSON
✅ 02. POST en texto plano se rechaza con 415
✅ 03. GET que pide XML se rechaza con 406
✅ 04. GET que acepta JSON responde 200
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion03.md` a `auditorias/auditoria_sesion03.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Me costó comprender cómo validar los encabezados HTTP, porque tuve que distinguir entre el formato de los datos que envío y el formato que acepta el cliente.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar un modelo para validar los datos de cada libro y una lista para guardar los libros creados temporalmente.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   Esta respuesta depende del resultado real de tu verificador. Si no detectó errores, puedes escribir:
   No se presentaron errores inesperados, porque las pruebas permitieron comprobar que la API respondiera con los códigos HTTP solicitados
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
